In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
print("Dataset downloaded to:", path)

import os
os.listdir(path)

In [ ]:
import pandas as pd

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(df.shape)
df.head()

In [ ]:
import re

def normalize_symptom(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

label_col = df.columns[0]
SYMPTOM_COLS = [c for c in df.columns if c != label_col]

df[label_col] = df[label_col].astype(str).str.strip()

symptom_vocab = [normalize_symptom(c) for c in SYMPTOM_COLS]
df = df.rename(columns=dict(zip(SYMPTOM_COLS, symptom_vocab)))

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

X = df[symptom_vocab].to_numpy(dtype=np.float32)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[label_col])

In [ ]:
from sklearn.model_selection import train_test_split

MIN_SAMPLES_PER_CLASS = 5

class_counts = pd.Series(y).value_counts()
keep_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
keep_mask = pd.Series(y).isin(keep_classes).to_numpy()

X_filtered = X[keep_mask]
y_filtered = y[keep_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

In [ ]:
from scipy.sparse import csr_matrix

X_train_sparse = csr_matrix(X_train)
X_test_sparse = csr_matrix(X_test)

xgb_classes = sorted(set(y_train) | set(y_test))
old_to_new = {old: new for new, old in enumerate(xgb_classes)}
new_to_old = {new: old for old, new in old_to_new.items()}

y_train_xgb = np.array([old_to_new[v] for v in y_train])
y_test_xgb = np.array([old_to_new[v] for v in y_test])

print(f"{len(xgb_classes)} classes remapped to a dense 0..{len(xgb_classes)-1} range")

In [ ]:
!pip install -q xgboost
import xgboost as xgb

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(xgb_classes),
    tree_method="hist",     
    max_depth=8,
    n_estimators=150,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=2,               
    random_state=42,
    eval_metric="mlogloss",
)

model.fit(
    X_train_sparse, y_train_xgb,
    eval_set=[(X_test_sparse, y_test_xgb)],
    verbose=20,
)
print("Trained.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_xgb = model.predict(X_test_sparse)
y_pred = np.array([new_to_old[v] for v in y_pred_xgb])  # map back for reporting

v3_accuracy = accuracy_score(y_test, y_pred)
v2_accuracy = 0.8521  # replace with your actual v2 number
v1_accuracy = 0.8162  # replace with your actual v1 number
print(f"v3 (XGBoost) accuracy:       {v3_accuracy:.4f}")
print(f"v2 (Random Forest) accuracy: {v2_accuracy:.4f}")
print(f"v1 (Decision Tree) accuracy: {v1_accuracy:.4f}")
print(f"v3 vs v2: {v3_accuracy - v2_accuracy:+.4f}")
print()

present_labels = sorted(set(y_test) | set(y_pred))
present_names = [label_encoder.classes_[i] for i in present_labels]
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

importances = pd.Series(model.feature_importances_, index=symptom_vocab).sort_values(ascending=False)
print("Top 20 most important symptoms:")
print(importances.head(20))

In [ ]:
import json
import joblib

os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(model, "model_artifacts/v3_xgboost.joblib")

# IMPORTANT: label_classes must follow XGBoost's dense 0..N-1 ordering
# (new_to_old), not the original encoder's indices — predict_proba()
# column i corresponds to dense class i, not the original label id.
label_classes = [label_encoder.classes_[new_to_old[i]] for i in range(len(xgb_classes))]
with open("model_artifacts/label_classes.json", "w") as f:
    json.dump(label_classes, f, indent=2)

with open("model_artifacts/symptom_vocab.json", "w") as f:
    json.dump(symptom_vocab, f, indent=2)

assert len(label_classes) == len(xgb_classes)
assert model.predict_proba(X_test_sparse[:1]).shape[1] == len(label_classes), "predict_proba width mismatch — do not ship"

print(f"{len(label_classes)} disease classes exported")
!ls -la model_artifacts

import shutil
shutil.make_archive("v3_model_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v3_model_artifacts.zip")